In [5]:
import pandas as pd
import plotly.express as px
import joblib

In [6]:
df = pd.read_csv("processed_user_risk_data.csv")

df.head()

,user_id,role,resource_type,action,timestamp,session_duration,access_volume,success_flag,assigned_resource_count,actively_used_resource_count,...,privilege_usage_ratio_zscore,resource_access_concentration_zscore,raw_risk_score,governance_risk_score,risk_category,anomaly_score,anomaly_label,ml_risk_score,pca1,pca2
0,USER_045,Cloud_Admin,gcp_console,read,2024-01-01 00:35:20,42.0,17.0,True,9,8,...,1.138516,-0.100936,0.342899,64.831138,High,0.071284,1,62.867399,3.202101,1.187607
1,USER_055,Cloud_Admin,iam_console,write,2024-01-01 00:57:24,57.0,14.0,True,7,5,...,0.256122,-0.183580,-0.212430,32.561941,Medium,0.112437,1,82.521392,-0.516974,0.605683
2,USER_064,Security_Admin,firewall_portal,write,2024-01-01 01:32:34,61.0,20.0,True,7,5,...,0.529781,0.542003,0.218229,57.586771,Medium,0.035352,1,45.706843,-0.070460,0.746893
3,USER_024,HR_Admin,time_tracking,read,2024-01-01 01:54:39,31.0,7.0,True,5,4,...,0.956613,-0.570736,-0.307262,27.051393,Low,0.046428,1,50.996430,-2.288915,3.270600
4,USER_044,Cloud_Admin,gcp_console,export,2024-01-01 02:38:32,75.0,16.0,True,9,7,...,0.576993,0.157172,0.132082,52.580917,Medium,0.113562,1,83.058354,1.507371,0.770607


In [7]:
model = joblib.load("isolation_forest_model.pkl")
scaler = joblib.load("scaler.pkl")

In [8]:
df['anomaly_label_str'] = df['anomaly_label'].map({
    1: 'Normal',
    -1: 'Anomaly'
})

In [9]:
fig = px.histogram(
    df,
    x="ml_risk_score",
    nbins=30,
    title="Distribution of ML Risk Scores"
)

fig.show()

In [10]:
fig = px.scatter(
    df,
    x="governance_risk_score",
    y="ml_risk_score",
    hover_data=["user_id"],
    title="ML vs Statistical Risk"
)

fig.show()

In [11]:
fig = px.scatter(
    df,
    x="pca1",
    y="pca2",
    color="anomaly_label_str",
    hover_data=["user_id", "ml_risk_score"],
    title="PCA Visualization of Anomalies"
)

fig.show()

In [12]:
top_users = df.sort_values(by='ml_risk_score', ascending=False).head(10)

fig = px.bar(
    top_users,
    x="user_id",
    y="ml_risk_score",
    title="Top 10 High Risk Users"
)

fig.show()

In [13]:
fig = px.box(
    df,
    y="ml_risk_score",
    title="Outlier Detection in Risk Scores"
)

fig.show()

In [14]:
fig = px.histogram(
    df,
    x="anomaly_label_str",
    title="Normal vs Anomalous Users"
)

fig.show()

In [15]:
print("=== ANOMALY LABEL DISTRIBUTION ===")
print(df['anomaly_label'].value_counts())
print(f"\nAnomaly %: {(df['anomaly_label'] == -1).sum() / len(df) * 100:.2f}%")

print("\n=== USER_055 INVESTIGATION ===")
print(df[df['user_id'] == 'USER_055']['ml_risk_score'].describe())
print(f"\nUSER_055 events: {(df['user_id'] == 'USER_055').sum()}")
print(f"Total events: {len(df)}")

print("\n=== TOP 10 USERS BY EVENT COUNT ===")
print(df['user_id'].value_counts().head(10))

print("\n=== ML RISK SCORE DISTRIBUTION ===")
print(df['ml_risk_score'].describe())

=== ANOMALY LABEL DISTRIBUTION ===
anomaly_label
 1    12031
-1      634
Name: count, dtype: int64

Anomaly %: 5.01%

=== USER_055 INVESTIGATION ===
count    123.000000
mean      90.336183
std        7.020870
min       65.037055
25%       86.205800
50%       90.940176
75%       95.940617
max      100.000000
Name: ml_risk_score, dtype: float64

USER_055 events: 123
Total events: 12665

=== TOP 10 USERS BY EVENT COUNT ===
user_id
USER_011    162
USER_010    159
USER_068    158
USER_059    157
USER_089    157
USER_081    156
USER_079    156
USER_030    155
USER_008    155
USER_019    154
Name: count, dtype: int64

=== ML RISK SCORE DISTRIBUTION ===
count    12665.000000
mean        60.735809
std         17.718196
min          0.000000
25%         47.508295
50%         63.454669
75%         74.472635
max        100.000000
Name: ml_risk_score, dtype: float64
